In [ ]:
# YOLOv8 Object Detection - Betel Insects Dataset
# Classes: Green-Leafhopper, aphid, beetle, grasshopper
# Features: Training, Evaluation, OOD Detection (Mahalanobis Distance)
# Platform: Kaggle Notebook
# =============================================================================

# ─────────────────────────────────────────────
# CELL 1: Install Dependencies
# ─────────────────────────────────────────────
!pip install ultralytics onnx onnxruntime onnxscript -q
import os, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
import yaml
import cv2
import torch
from collections import defaultdict

from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.covariance import EmpiricalCovariance

warnings.filterwarnings("ignore")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Configuration
# ─────────────────────────────────────────────

# Dataset & output paths
DATASET_ROOT   = Path("/kaggle/input/datasets/amandapriyashantha/betel-pets-obj-01")
WORKING_DIR    = Path("/kaggle/working")
FILTERED_DIR   = WORKING_DIR / "betel_filtered"
OUTPUT_DIR     = WORKING_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Classes to KEEP (indices in the ORIGINAL dataset must be discovered first)
KEEP_CLASSES   = ["Green-Leafhopper", "aphid", "beetle", "grasshopper"]
REMOVE_CLASSES = ["armyworm", "sawfly"]

# Training hyper-parameters
IMG_SIZE  = 640
EPOCHS    = 25
BATCH     = 16
DEVICE    = 0 if torch.cuda.is_available() else "cpu"
MODEL_CFG = "yolov8m.pt"           # medium backbone – good balance for Kaggle P100

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device         : {DEVICE}")
print(f"Dataset root   : {DATASET_ROOT}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Discover Original Class Mapping
# ─────────────────────────────────────────────

orig_yaml_path = DATASET_ROOT / "data.yaml"
with open(orig_yaml_path) as f:
    orig_cfg = yaml.safe_load(f)

orig_classes = orig_cfg["names"]           # list of class names
print("Original classes:", orig_classes)

# Map: original index → class name
orig_id2name = {i: n for i, n in enumerate(orig_classes)}
orig_name2id = {n: i for i, n in enumerate(orig_classes)}

# New consecutive mapping (keep order)
new_classes  = [c for c in orig_classes if c in KEEP_CLASSES]
new_name2id  = {n: i for i, n in enumerate(new_classes)}
new_id2name  = {i: n for i, n in enumerate(new_classes)}

# Set of original IDs to KEEP
keep_orig_ids = {orig_name2id[c] for c in KEEP_CLASSES if c in orig_name2id}
# Remap: original_id → new_id
remap = {oid: new_name2id[orig_id2name[oid]] for oid in keep_orig_ids}

print("New classes    :", new_classes)
print("Remap table    :", remap)

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 (UPDATED): Build dataset from train folder only
#   - Cap 2400 images per class
#   - Split → 80% train / 15% val / 5% test
# ─────────────────────────────────────────────
 
import shutil, random
from collections import Counter, defaultdict
from pathlib import Path
 
MAX_PER_CLASS  = 2400
TRAIN_RATIO    = 0.80
VAL_RATIO      = 0.15
TEST_RATIO     = 0.05   # remainder after train+val
random.seed(42)
 
# Clean and recreate output split dirs
for split in ["train", "valid", "test"]:
    for sub in ["images", "labels"]:
        d = FILTERED_DIR / split / sub
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)
 
 
# ── Step 1: Scan ONLY the original train folder ──────────────────────
src_img_dir = DATASET_ROOT / "train" / "images"
src_lbl_dir = DATASET_ROOT / "train" / "labels"
 
candidates = []   # (img_path, lbl_path, primary_new_class_id, all_new_class_ids)
 
for img_path in sorted(src_img_dir.iterdir()):
    if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
        continue
    lbl_path = src_lbl_dir / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue
 
    new_ids = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                orig_cid = int(parts[0])
                if orig_cid in keep_orig_ids:
                    new_ids.append(remap[orig_cid])
 
    if not new_ids:
        continue   # image has no kept-class annotations
 
    primary = Counter(new_ids).most_common(1)[0][0]
    candidates.append((img_path, lbl_path, primary, new_ids))
 
print(f"Total candidate images (all classes): {len(candidates)}")
 
# Raw annotation counts before capping
raw_ann = defaultdict(int)
for _, _, _, ids in candidates:
    for i in ids:
        raw_ann[i] += 1
print("Raw annotation counts per class:")
for cid in sorted(raw_ann):
    print(f"  {new_id2name[cid]:20s}: {raw_ann[cid]} annotations")
 
 
# ── Step 2: Cap at MAX_PER_CLASS per primary class ───────────────────
random.shuffle(candidates)
 
class_counts = defaultdict(int)
capped = []
for item in candidates:
    _, _, primary, _ = item
    if class_counts[primary] < MAX_PER_CLASS:
        capped.append(item)
        class_counts[primary] += 1
 
print(f"\nAfter capping at {MAX_PER_CLASS}/class:")
for cid in sorted(class_counts):
    print(f"  {new_id2name[cid]:20s}: {class_counts[cid]} images")
print(f"  Total selected         : {len(capped)} images")
 
 
# ── Step 3: Stratified split per class → train / val / test ──────────
# Group by primary class, split each group, then merge
 
train_items, val_items, test_items = [], [], []
 
for cid in sorted(class_counts):
    class_pool = [item for item in capped if item[2] == cid]
    random.shuffle(class_pool)
    n          = len(class_pool)
    n_val      = round(n * VAL_RATIO)
    n_test     = round(n * TEST_RATIO)
    n_train    = n - n_val - n_test
 
    train_items.extend(class_pool[:n_train])
    val_items.extend(class_pool[n_train : n_train + n_val])
    test_items.extend(class_pool[n_train + n_val :])
 
    print(f"  {new_id2name[cid]:20s}  "
          f"train={n_train}  val={n_val}  test={n_test}")
 
print(f"\nFinal split totals → "
      f"train={len(train_items)}  val={len(val_items)}  test={len(test_items)}")
 
 
# ── Step 4: Helper — write remapped label and copy image ─────────────
def write_split_item(img_path: Path, lbl_path: Path, split: str):
    """Remap label file and copy image into the filtered split folder."""
    dst_lbl = FILTERED_DIR / split / "labels" / (img_path.stem + ".txt")
    dst_img = FILTERED_DIR / split / "images" / img_path.name
 
    lines_out = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            orig_cid = int(parts[0])
            if orig_cid in keep_orig_ids:
                new_id = remap[orig_cid]
                lines_out.append(f"{new_id} " + " ".join(parts[1:]))
 
    if not lines_out:
        return False
 
    with open(dst_lbl, "w") as f:
        f.write("\n".join(lines_out) + "\n")
    shutil.copy2(img_path, dst_img)
    return True
 
 
# ── Step 5: Write all splits ─────────────────────────────────────────
for split, items in [("train", train_items),
                     ("valid",  val_items),
                     ("test",   test_items)]:
    written = 0
    for img_path, lbl_path, _, _ in items:
        if write_split_item(img_path, lbl_path, split):
            written += 1
    print(f"  [{split}] wrote {written} images")
 
# ── Step 6: Annotation counts per split (verification) ───────────────
print("\nAnnotation counts per class per split:")
for split in ["train", "valid", "test"]:
    lbl_dir = FILTERED_DIR / split / "labels"
    ann_counts = defaultdict(int)
    for lbl_file in lbl_dir.glob("*.txt"):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    ann_counts[int(parts[0])] += 1
    print(f"  [{split}]  " +
          "  ".join(f"{new_id2name[c]}={ann_counts[c]}"
                    for c in sorted(ann_counts)))
 
print("\nDone.")
